# Lista 05 - Regras de associação no Titanic

Solução da Questão 07 usando Apriori. O notebook trata a base, cria transações com presença e ausência, compara parâmetros e analisa cinco regras concretas.

## 1. Leitura e bibliotecas

A biblioteca `apyori` é a mesma utilizada no notebook-base da disciplina.

In [ ]:
!pip install -q apyori
import numpy as np
import pandas as pd
from apyori import apriori
from IPython.display import display

CAMINHO = 'titanic completo.csv'
original = pd.read_csv(CAMINHO)
print('Dimensões:', original.shape)
display(original.head())

## 2. Pré-processamento

Foram removidos nome, ticket, cabine, barco, corpo e destino de origem. Identificadores não generalizam bem; barco e corpo revelam o resultado e causariam vazamento. Ausências numéricas são preenchidas pela mediana e embarque pela moda.

A idade usa as faixas 0-17, 18-22, 23-40 e 41+. A tarifa usa 0-10, 10-30, 30-80 e 80+, faixas que facilitam a interpretação.

In [ ]:
base = original.copy()
base['age'] = base['age'].fillna(base['age'].median())
base['fare'] = base['fare'].fillna(base['fare'].median())
base['embarked'] = base['embarked'].fillna(base['embarked'].mode()[0])

base['Classe'] = base['pclass'].map({1:'1a', 2:'2a', 3:'3a'})
base['Sexo'] = base['sex'].map({'female':'Feminino', 'male':'Masculino'})
base['Idade'] = pd.cut(base['age'], [0,17,22,40,np.inf], labels=['0-17','18-22','23-40','41+'], include_lowest=True).astype(str)
base['Tarifa'] = pd.cut(base['fare'], [-np.inf,10,30,80,np.inf], labels=['0-10','10-30','30-80','80+']).astype(str)
base['Embarque'] = base['embarked'].map({'C':'Cherbourg','Q':'Queenstown','S':'Southampton'})
base['Irmãos/cônjuges'] = pd.cut(base['sibsp'], [-1,0,1,np.inf], labels=['0','1','2+']).astype(str)
base['Pais/filhos'] = pd.cut(base['parch'], [-1,0,1,np.inf], labels=['0','1','2+']).astype(str)
base['Sobreviveu'] = base['survived'].map({0:'Não', 1:'Sim'})

atributos = ['Classe','Sexo','Idade','Tarifa','Embarque','Irmãos/cônjuges','Pais/filhos','Sobreviveu']
dados = base[atributos].copy()
print('Ausências restantes:', int(dados.isna().sum().sum()))
display(dados.head())

## 3. Transações com presença e ausência

Cada passageiro vira uma transação. Para cada atributo preditor, entram o item presente e os itens `NÃO atributo=valor` para as categorias que não estão presentes. Por exemplo, um passageiro da primeira classe recebe `Classe=1a`, `NÃO Classe=2a` e `NÃO Classe=3a`.

Não é criada ausência para Sobreviveu: isso evita regras tautológicas como `NÃO Sobreviveu=Não -> Sobreviveu=Sim`.

In [ ]:
preditores = [a for a in atributos if a != 'Sobreviveu']
categorias = {a: sorted(dados[a].unique()) for a in preditores}
transacoes = []
for _, linha in dados.iterrows():
    transacao = []
    for atributo in preditores:
        valor = linha[atributo]
        transacao.append(f'{atributo}={valor}')
        transacao.extend(f'NÃO {atributo}={outro}' for outro in categorias[atributo] if outro != valor)
    transacao.append(f'Sobreviveu={linha["Sobreviveu"]}')
    transacoes.append(transacao)
print('Quantidade de transações:', len(transacoes))
print('Exemplo:', transacoes[0])

## 4. Teste de parâmetros

Os limiares são comparados para justificar a escolha. Foi escolhida a configuração com suporte mínimo 0.20 e confiança mínima 0.75, pois reduz regras muito raras e ainda preserva associações interpretáveis.

In [ ]:
configuracoes = [(0.15,0.70),(0.20,0.75),(0.25,0.80),(0.30,0.80),(0.20,0.70)]
saidas = {}
comparacao = []
for suporte_min, confianca_min in configuracoes:
    saida_atual = list(apriori(transacoes, min_support=suporte_min, min_confidence=confianca_min, min_lift=1.0, min_length=2))
    saidas[(suporte_min, confianca_min)] = saida_atual
    quantidade = sum(len(itemset.ordered_statistics) for itemset in saida_atual)
    comparacao.append({'min_support': suporte_min, 'min_confidence': confianca_min, 'itemsets_frequentes': len(saida_atual), 'regras': quantidade})
display(pd.DataFrame(comparacao))

## 5. Regras e métricas

Para cada regra são exibidos suporte, confiança e lift. Regras com lift menor ou igual a 1 são retiradas da análise principal por não indicarem associação positiva.

In [ ]:
MIN_SUPPORT, MIN_CONFIDENCE = 0.20, 0.75
saida = saidas[(MIN_SUPPORT, MIN_CONFIDENCE)]
linhas = []
for itemset in saida:
    for estatistica in itemset.ordered_statistics:
        ant = sorted(estatistica.items_base)
        cons = sorted(estatistica.items_add)
        if ant and cons:
            linhas.append({'Antecedente': ' ; '.join(ant), 'Consequente': ' ; '.join(cons), 'Suporte': itemset.support, 'Confiança': estatistica.confidence, 'Lift': estatistica.lift})
regras = pd.DataFrame(linhas).drop_duplicates()
regras = regras[regras['Lift'] > 1].sort_values(['Lift','Confiança','Suporte'], ascending=False).reset_index(drop=True)
print('Regras não triviais:', len(regras))
display(regras.head(30).style.format({'Suporte':'{:.3f}','Confiança':'{:.3f}','Lift':'{:.3f}'}))

## 6. Separação por sobrevivência

Estas tabelas atendem ao item 4b da lista. O filtro considera regras cujo consequente é exatamente a sobrevivência, evitando misturar outra característica no consequente.

In [ ]:
regras_sim = regras[regras['Consequente'].eq('Sobreviveu=Sim')].copy()
regras_nao = regras[regras['Consequente'].eq('Sobreviveu=Não')].copy()
print('Regras com consequente Sobreviveu=Sim:', len(regras_sim))
display(regras_sim.head(15).style.format({'Suporte':'{:.3f}','Confiança':'{:.3f}','Lift':'{:.3f}'}))
print('Regras com consequente Sobreviveu=Não:', len(regras_nao))
display(regras_nao.head(15).style.format({'Suporte':'{:.3f}','Confiança':'{:.3f}','Lift':'{:.3f}'}))

## 7. Regras com ausência

A tabela abaixo evidencia as regras que usam `NÃO ...` no antecedente, conforme solicitado no item 2.

In [ ]:
regras_ausencia = regras[regras['Antecedente'].str.contains('NÃO ', regex=False)].copy()
display(regras_ausencia.head(20).style.format({'Suporte':'{:.3f}','Confiança':'{:.3f}','Lift':'{:.3f}'}))

## 8. Cinco interpretações concretas

A célula seguinte seleciona cinco regras reais da execução, garante pelo menos duas com ausência e imprime uma interpretação com os valores calculados.

In [ ]:
candidatas = regras[regras['Consequente'].isin(['Sobreviveu=Sim','Sobreviveu=Não'])].copy()
candidatas['tem_ausencia'] = candidatas['Antecedente'].str.contains('NÃO ', regex=False)
candidatas = candidatas.sort_values(['Lift','Confiança','Suporte'], ascending=False)
selecionadas = []
for _, linha in candidatas[candidatas['tem_ausencia'] & candidatas['Consequente'].eq('Sobreviveu=Sim')].iterrows():
    if len(selecionadas) < 2: selecionadas.append(linha)
for _, linha in candidatas.iterrows():
    if len(selecionadas) >= 5: break
    if not any(linha['Antecedente'] == x['Antecedente'] and linha['Consequente'] == x['Consequente'] for x in selecionadas): selecionadas.append(linha)
analise_5 = pd.DataFrame(selecionadas).head(5)
display(analise_5.drop(columns=['tem_ausencia'], errors='ignore').style.format({'Suporte':'{:.3f}','Confiança':'{:.3f}','Lift':'{:.3f}'}))
for numero, (_, linha) in enumerate(analise_5.iterrows(), start=1):
    tipo = 'sobrevivência' if linha['Consequente'] == 'Sobreviveu=Sim' else 'não sobrevivência'
    extra = ' Possui ausência explícita no antecedente: ' + linha['Antecedente'] + '.' if 'NÃO ' in linha['Antecedente'] else ''
    print('Regra {}: {} -> {} | suporte={:.3f}, confiança={:.3f}, lift={:.3f}. Associação com {}.{}'.format(numero, linha['Antecedente'], linha['Consequente'], linha['Suporte'], linha['Confiança'], linha['Lift'], tipo, extra))

## Conclusão

As regras não são causais: descrevem associações observadas. Em geral, regras de sobrevivência destacam grupos femininos e de maior tarifa ou classe, enquanto regras de não sobrevivência tendem a destacar homens e tarifas menores. A análise considera simultaneamente suporte, confiança e lift.